# A handful of hidden taste dials

Unit 4 asked *readers who liked the books you liked also read what?* and answered it by **memorizing** co-occurrence: one similarity number for every pair of books, read straight off who-read-what. It was the widest win so far (about 0.252 hit@10, roughly 2.3x popularity and 1.6x the lexical path) -- but it has a hard ceiling. Two books no single reader has ever read *together* score exactly zero as neighbours, however alike their readers' tastes really are, and about 41% of the catalog (818 books) no one has co-read at all, so CF is blind to it.

This unit asks a bolder question. *What if we could describe every reader and every book by a handful of hidden "taste dials"* -- say, "how science-fiction is this reader?", "how romance is this book?" -- and score a reader-book pair by how well their dials line up? We never get told what the dials are. We **learn** them from the same interaction log, by gradient descent, so that the dials of a reader and the dials of the books they liked point the same way.

That is **matrix factorization** (MF), and it is the Part-1 -> Part-2 hinge of this book. The dials we learn are **latent factors** -- and a learned vector per reader and per book, scored by a dot product, is exactly an **embedding**. The same shape Unit 8's neural two-tower will learn. We will build the model from scratch in numpy, reveal the shipped `MatrixFactorizationPath`, and measure it: on this data MF comes out **on par with item-item CF** (about 0.25-0.28 across seeds, within noise -- here it prints 0.276), the *latent generalization* of co-occurrence, while compressing the whole signal into 32 numbers per reader and per book.

## Load the catalog and the interaction log

The Unit 1-4 substrate is unchanged. `bookrec.generated_dir()` locates the seeded data slice (run the recsys generators first if it complains), `load_catalog` gives the `dict[int, Book]`, and the interaction log is the gzip-ed CSV with one row per reader-item event: `reader_id, item_id, split, label`. We load the keyword table and the held-out `cold_readers` too, so the scoreboard at the end lines MF up against every earlier path under the exact Unit 1 rules.

In [ ]:
import json

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])

print(f"catalog: {len(catalog_ids)} books; interaction log: {len(interactions)} rows")
print(f"cold readers held out of scoring: {len(cold_readers)}")

## 1. The latent-factor model: $P\,Q^{\top}$

Give every reader a short vector of $f$ numbers and every book a short vector of the *same* $f$ numbers. Stack the reader vectors as the rows of a matrix $\mathbf{P}$ (readers $\times$ factors) and the book vectors as the rows of $\mathbf{Q}$ (books $\times$ factors). Then one reader's predicted affinity for one book is just the **dot product** of their two vectors:

$$\mathrm{score}(u, i) = \mathbf{P}_u \cdot \mathbf{Q}_i = \sum_{d=1}^{f} \mathbf{P}_{u,d}\,\mathbf{Q}_{i,d}$$

and the whole reader $\times$ book score table is the single matrix product $\mathbf{P}\,\mathbf{Q}^{\top}$. That is the entire model. Each of the $f$ coordinates is a **latent factor** -- a hidden "taste dial." We never name the dials; we only hope that, once trained, coordinate $d$ of a reader and coordinate $d$ of a book measure *the same* hidden quality, so their product is large exactly when reader and book agree on it.

Here is the idea on two dials we *do* name, by hand, just to see the shape. Suppose dial 0 is "science-fiction-ness" and dial 1 is "romance-ness." A reader who loves sci-fi and dislikes romance is $[\,2, -1\,]$; a hard sci-fi book is $[\,2, 0\,]$, a romance $[\,0, 2\,]$. The dot product scores each pairing without anyone writing a rule.

In [ ]:
reader = np.array([2.0, -1.0])            # [sci-fi-ness, romance-ness] of one reader
books = {
    "hard sci-fi":   np.array([2.0, 0.0]),
    "space romance": np.array([1.5, 1.5]),
    "pure romance":  np.array([0.0, 2.0]),
}
print(f"{'book':>14}  {'factors':>12}  score = reader . book")
for title, q in books.items():
    print(f"{title:>14}  {str(q):>12}  {float(reader @ q):+.1f}")
print("\nThe reader's dials rank hard sci-fi top and pure romance bottom -- no genre rule, just a dot product.")

### A short aside: why not plain SVD?

If you have met the **singular value decomposition**, you may object that linear algebra *already* factors a matrix into low-rank pieces -- and truncated SVD even runs on sparse matrices. Why train with gradient descent instead of just decomposing the reader $\times$ book matrix?

Because of what the two methods do with the **unobserved** entries. Truncated SVD minimizes the *squared reconstruction error over every entry of the matrix* -- so it treats each reader-book pair we never saw as a hard **0** it must reproduce, i.e. as a confirmed *dislike*. But our feedback is **implicit**: a read or a like is a positive, and a pair we never saw is *unknown*, not a dislike (the reader may simply never have been shown the book). Forcing all those unknowns to 0 is the wrong objective. So instead we optimize a **logistic** loss over the observed positives **plus a sample of negatives**, and fit the factors by gradient descent. That is why the MF here is *trained*, not *decomposed* -- and why we build it as a training loop below rather than calling an SVD routine.

## 2. The logistic loss, its gradient, and training by hand

We want $\mathrm{sigmoid}(\mathbf{P}_u \cdot \mathbf{Q}_i)$ to be near **1** for a pair the reader liked and near **0** for a negative, where $\mathrm{sigmoid}(x) = 1/(1 + e^{-x})$ squashes any score into $(0, 1)$. The natural cost for that is the **logistic (binary cross-entropy) loss**. For a single pair with label $y \in \{0, 1\}$ and prediction $\hat{y} = \mathrm{sigmoid}(\mathbf{P}_u \cdot \mathbf{Q}_i)$,

$$\ell(u, i) = -\bigl[\,y\,\log \hat{y} + (1 - y)\,\log(1 - \hat{y})\,\bigr].$$

Its gradients have an almost magically clean form. Writing the **error** $e = y - \hat{y}$,

$$\frac{\partial \ell}{\partial \mathbf{P}_u} = -e\,\mathbf{Q}_i, \qquad \frac{\partial \ell}{\partial \mathbf{Q}_i} = -e\,\mathbf{P}_u.$$

So each training pair nudges a reader's vector toward the books it should like and away from its negatives, and vice versa. We descend with three refinements:

- **Full-batch**: sum every pair's gradient contribution onto the reader (or book) it touches, in one vectorized pass per epoch (no per-pair Python loop -- 58k pairs x 300 epochs would be far too slow).
- **Per-entity averaging**: divide each entity's summed gradient by *its own* interaction count, so a reader with many positives takes a step the same size as a reader with few (the step is well-scaled regardless of how busy a reader or book is).
- **L2 regularization**: pull every vector gently back toward zero each step (strength `reg`), so the factors do not overfit the handful of positives we have.

The per-entity sum is the one trick worth seeing: we bucket each pair's gradient row into its entity with a `bincount` per factor column -- far faster than a Python scatter. Here is the whole trainer.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def per_entity_sum(index, grad_rows, n):
    # Sum grad_rows[j] into bucket index[j]; one bincount per factor column (fast scatter-add).
    out = np.empty((n, grad_rows.shape[1]))
    for d in range(grad_rows.shape[1]):
        out[:, d] = np.bincount(index, weights=grad_rows[:, d], minlength=n)
    return out


def train_factors(pos_r, pos_i, neg_r, neg_i, n_readers, n_items,
                  n_factors=32, n_epochs=300, lr=0.5, reg=0.05, seed=0, report=0):
    # Full-batch, per-entity-averaged gradient descent on the logistic implicit-feedback loss.
    rng = np.random.default_rng(seed)
    P = rng.normal(0.0, 0.1, size=(n_readers, n_factors))        # seeded factor init
    Q = rng.normal(0.0, 0.1, size=(n_items, n_factors))
    u = np.concatenate([pos_r, neg_r])                           # every training pair, positives then negatives
    i = np.concatenate([pos_i, neg_i])
    y = np.concatenate([np.ones(len(pos_r)), np.zeros(len(neg_r))])
    count_u = np.maximum(np.bincount(u, minlength=n_readers), 1)[:, None]   # per-entity interaction counts
    count_i = np.maximum(np.bincount(i, minlength=n_items), 1)[:, None]
    for epoch in range(n_epochs):
        err = (y - sigmoid((P[u] * Q[i]).sum(axis=1)))[:, None]  # e = y - yhat, per pair
        grad_P = per_entity_sum(u, err * Q[i], n_readers)        # sum e*Q onto each reader
        grad_Q = per_entity_sum(i, err * P[u], n_items)          # sum e*P onto each book
        P += lr * (grad_P / count_u - reg * P)                   # average by count, pull back by reg
        Q += lr * (grad_Q / count_i - reg * Q)
        if report and (epoch == 0 or (epoch + 1) % report == 0):
            fit = -np.log(sigmoid((P[pos_r] * Q[pos_i]).sum(axis=1))).mean()
            print(f"  epoch {epoch + 1:>3}: mean positive logistic loss = {fit:.3f}")
    return P, Q

Watch it learn on a tiny toy with two taste clusters: readers 0, 1, 2 each liked books 0, 1, 2; readers 3, 4, 5 each liked books 3, 4, 5. For negatives we hand each reader the *other* cluster's books (a crisp stand-in for "unobserved" here). If training works, the mean positive loss should fall epoch over epoch as each reader's vector swings toward its own cluster.

In [ ]:
pos_r = np.array([r for r in (0, 1, 2) for _ in range(3)] + [r for r in (3, 4, 5) for _ in range(3)])
pos_i = np.array([i for _ in (0, 1, 2) for i in (0, 1, 2)] + [i for _ in (3, 4, 5) for i in (3, 4, 5)])
neg_r = np.array([r for r in (0, 1, 2) for _ in range(3)] + [r for r in (3, 4, 5) for _ in range(3)])
neg_i = np.array([i for _ in (0, 1, 2) for i in (3, 4, 5)] + [i for _ in (3, 4, 5) for i in (0, 1, 2)])

print("training two taste clusters by hand (6 readers, 6 books, 4 factors):")
toy_P, toy_Q = train_factors(pos_r, pos_i, neg_r, neg_i, 6, 6, n_factors=4, n_epochs=200, seed=0, report=50)
print(f"\nreader 0 now scores its own book 0 at {float(toy_P[0] @ toy_Q[0]):+.2f} "
      f"and the cross-cluster book 3 at {float(toy_P[0] @ toy_Q[3]):+.2f}")

### Reveal `MatrixFactorizationPath`

The package ships exactly this training loop as a standard retrieval path, with one piece done for you: it **samples** the negatives (we will see which negatives in a moment) instead of taking them as an argument. `bookrec.MatrixFactorizationPath` fits `P` / `Q` from the **train positives** of the row-mapping log (same `fit(interactions, catalog=None)` signature as every other path -- no pandas inside the package), then `retrieve(query, context, k)` scores a fitted reader's whole catalog by $\mathbf{P}_u \cdot \mathbf{Q}^{\top}$ and returns the top `k` unseen books.

Its defaults are **pinned** to the configuration measured on this data -- `n_factors=32, n_epochs=300, learning_rate=0.5, reg=0.05, n_negatives=10` -- and they are *not* "small/fast": below about 300 epochs MF quietly slips under the lexical path, so do not shrink them. Confirm on the toy that the shipped path drives the same positive loss down, then fit it on the real log (about 15-20 seconds).

In [ ]:
toy_rows = [{"reader_id": int(r), "item_id": int(i), "split": "train", "label": 1}
            for r, i in zip(pos_r, pos_i)]
toy_catalog = [0, 1, 2, 3, 4, 5]


def positive_loss(path, pairs):
    art = path.artifact()
    P, Q = art["P"], art["Q"]
    r_of = {r: j for j, r in enumerate(art["reader_ids"])}
    i_of = {it: j for j, it in enumerate(art["item_ids"])}
    s = np.array([float(P[r_of[r]] @ Q[i_of[i]]) for r, i in pairs])
    return float(-np.log(sigmoid(s)).mean())


toy_pairs = list(zip([int(r) for r in pos_r], [int(i) for i in pos_i]))
barely = bookrec.MatrixFactorizationPath(n_epochs=1, seed=0).fit(toy_rows, catalog=toy_catalog)
trained = bookrec.MatrixFactorizationPath(n_epochs=300, seed=0).fit(toy_rows, catalog=toy_catalog)
print(f"shipped path on the toy: positive loss  1 epoch = {positive_loss(barely, toy_pairs):.3f}"
      f"   ->   300 epochs = {positive_loss(trained, toy_pairs):.3f}")

mf = bookrec.MatrixFactorizationPath(seed=0).fit(rows, catalog=catalog_ids)
P = mf.reader_factors
Q = mf.item_factors
print(f"\nfitted on the real log: P is {P.shape} (readers x factors), Q is {Q.shape} (books x factors)")
print(f"readers with a learned factor (>= 1 train positive): {len(mf.artifact()['reader_ids'])}")

## 2b. Which negatives? The beat Unit 4 set up

Unit 4 ended on a loose thread: the log's `label == 0` rows -- books a reader was **shown but did not engage** (exposure-sampled negatives) -- carried no co-*liking* signal for a counting path, "but they become training signal in Unit 5." Here we are, and the choice of negatives turns out to decide everything. There are two candidate pools:

- the log's **exposure negatives** (`label == 0` train rows): the specific books each reader saw and skipped;
- each reader's **unobserved complement**: books sampled uniformly from the catalog that the reader has *no* positive for (what `MatrixFactorizationPath` does internally).

They sound interchangeable. They are not. Train the exact same trainer both ways and measure on the validation scoreboard.

In [ ]:
def val_hit(path):
    board = bookrec.run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
    )
    return board.hit_rate_at_k


# Build the training arrays from the real log: positives, and the label==0 exposure negatives.
train_pos = [(r["reader_id"], r["item_id"]) for r in rows if r["split"] == "train" and r["label"] == 1]
exposure = [(r["reader_id"], r["item_id"]) for r in rows if r["split"] == "train" and r["label"] == 0]
readers = sorted({r for r, _ in train_pos})
reader_col = {r: j for j, r in enumerate(readers)}
item_col = {it: j for j, it in enumerate(catalog_ids)}
exposure = [(r, it) for r, it in exposure if r in reader_col]   # keep negs for readers that have a P row

pr = np.array([reader_col[r] for r, _ in train_pos])
pi = np.array([item_col[it] for _, it in train_pos])
er = np.array([reader_col[r] for r, _ in exposure])
ei = np.array([item_col[it] for _, it in exposure])

# Train on the EXPOSURE negatives, wrap the learned factors as a path, and score it.
Pe, Qe = train_factors(pr, pi, er, ei, len(readers), len(catalog_ids), seed=0)
exposure_mf = bookrec.MatrixFactorizationPath().load({
    "P": Pe, "Q": Qe, "reader_ids": readers, "item_ids": catalog_ids,
    "params": {"n_factors": 32, "n_epochs": 300, "learning_rate": 0.5,
               "reg": 0.05, "n_negatives": 10, "seed": 0},
})

mf_hit = val_hit(mf)                       # complement negatives (the shipped path)
exposure_hit = val_hit(exposure_mf)        # exposure negatives
print(f"MF trained on each reader's unobserved complement : hit@10 = {mf_hit:.4f}")
print(f"MF trained on the log's label==0 exposure negatives: hit@10 = {exposure_hit:.4f}")

The complement-trained model is the real recommender; the exposure-trained one **collapses to the random floor** (it prints about 0.01 -- at or below random's 0.012). Same loss, same optimizer, same positives: only the negatives changed.

Why is the "obvious" negative the poison one? Because the generator does not expose books at random -- it shows each reader books by **popularity and their own latent taste**. So a `label == 0` row is an *exposure-biased* negative: it is disproportionately a book the reader would plausibly *like* (that is why it got shown) but merely did not get to. Training the model to push those scores **down** tells it to dislike exactly the books it should recommend, destroying the signal. The complement, sampled uniformly from everything the reader has no positive for, is mostly genuine non-interest and keeps the signal intact.

This is your first concrete run-in with **exposure bias** -- the gap between what a reader *chose* and what they were merely *offered* -- which gets a unit of its own (Unit 13). For now the lesson is narrow and binding: in implicit feedback, *sample* your negatives from the complement; do not reach for the exposure negatives just because the log labels them 0.

## 3. Score it: the latent generalization of co-occurrence

Now the scoreboard, under the Unit 1 rules unchanged -- hit-rate@10 over positive `val` rows, each reader's train positives removed from both the recommendations and the relevance set, the 60 `cold_readers` excluded, the random floor seeded at 0. We line the complement-trained MF up against the random floor, popularity (Unit 2), lexical (Unit 3) and item-item CF (Unit 4).

In [ ]:
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=0)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
cf = bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)

registry = bookrec.PathRegistry()
for path in (random_floor, popularity, lexical, cf, mf):
    registry.register(path)

board = {
    "random": val_hit(random_floor),
    "popularity": val_hit(popularity),
    "lexical": val_hit(lexical),
    "item-item CF": val_hit(cf),
    "matrix factorization": mf_hit,
}
floor = board["random"]
print(f"{'path':>22} {'hit@10':>8} {'vs floor':>9}")
for label, hit in board.items():
    print(f"{label:>22} {hit:>8.4f} {hit / floor:>8.1f}x")
print(f"\nMF vs CF: {board['matrix factorization'] / board['item-item CF']:.2f}x   "
      f"MF vs lexical: {board['matrix factorization'] / board['lexical']:.2f}x   "
      f"MF vs popularity: {board['matrix factorization'] / board['popularity']:.2f}x")

### Read the scoreboard honestly

MF prints about **0.276** hit@10 here -- roughly 23x the random floor, 2.5x popularity (~0.108) and 1.75x the lexical path (~0.158). But do **not** read the single number as "MF beats CF." Across random seeds the negative sampling makes MF land anywhere in **about 0.25-0.28** (mean ~0.27); CF sits at ~0.252. The gap at this seed (+0.024) is **within the noise** of the sampling -- the honest headline is that **MF is on par with item-item CF**, not that it reliably beats it. More epochs pull MF slightly ahead, at a time cost.

That parity is itself the point. CF *memorizes* one similarity per observed pair and scores a never-co-read pair at exactly zero. MF reaches the same accuracy having **compressed the entire log into 32 numbers per reader and per book** -- and because two books that share *neighbours of neighbours* end up close in that 32-d space even when no single reader read both, MF is the **latent generalization of co-occurrence**. It recovers CF's signal without storing CF's giant pairwise table.

### The hinge: these factors ARE embeddings

Look again at what `retrieve` computes: it scores every book by $\mathbf{P}_u \cdot \mathbf{Q}_i$, a dot product of a learned reader vector against learned book vectors. That is precisely a **dot-product retriever over embeddings**. We learned a vector per reader and per book with gradient descent, and recommendation is nearest-neighbour search in that learned space. Confirm the identity directly -- the score `retrieve` returns is exactly $\mathbf{P}_u \cdot \mathbf{Q}_i$.

In [ ]:
reader_id = mf.artifact()["reader_ids"][0]
r_row = {r: j for j, r in enumerate(mf.artifact()["reader_ids"])}[reader_id]
scores = mf.reader_factors[r_row] @ mf.item_factors.T     # P_u . Q^T, the whole catalog at once

recs = mf.retrieve(reader_id, {"seen": set()}, 5)
print(f"top MF books for reader {reader_id}:")
print(f"{'book':>8} {'retrieve score':>15} {'P_u . Q_i':>12}")
for c in recs:
    col = item_col[c.item_id]
    assert abs(c.score - float(scores[col])) < 1e-9       # the score IS the embedding dot product
    print(f"{c.item_id:>8} {c.score:>15.4f} {float(scores[col]):>12.4f}")
print("\nretrieve score == P_u . Q_i exactly: a dot-product retriever over learned embeddings")
print("-> Unit 8 learns these two embedding tables with a neural two-tower instead of this loop.")

### The honest cold story (two different cases)

MF's reach is *broader* than CF's -- it gives every catalog book a `Q` row, so it can *score* all 2000 books, where CF scores 818 of them at a hard zero. But "scores" is not "recommends well," and cold-start splits into two genuinely different cases:

- A **cold reader** (0 train positives) never got a `P` row -- MF has no factor for them, so `retrieve` honestly returns `[]`. There is nothing to be similar *with*.
- A **cold item** (0 train positives) *does* get an initialized `Q` row, but it never appears as a positive. It only ever shows up as a sampled **negative**, so the gradient only ever pushes its score *down*. MF can score it, but it will rank near the bottom -- so MF still cannot meaningfully *recommend* a brand-new book.

See both: a held-out cold reader returns nothing, and the 818 cold books sit well below the warm books in a fitted reader's scores.

In [ ]:
known_readers = set(mf.artifact()["reader_ids"])
cold_reader = next(r for r in sorted(cold_readers) if r not in known_readers)
print(f"cold reader {cold_reader} (0 train positives) -> retrieve returns {mf.retrieve(cold_reader, {'seen': set()}, 5)}")

warm_items = {it for _, it in train_pos}
cold_cols = [item_col[it] for it in catalog_ids if it not in warm_items]
warm_cols = [item_col[it] for it in catalog_ids if it in warm_items]
print(f"\ncatalog books MF can score: {len(catalog_ids)} (every book gets a Q row)")
print(f"  of which cold (no train positive): {len(cold_cols)}  -- CF scores these at a hard zero")
print(f"mean MF score for reader {reader_id}:  warm books {scores[warm_cols].mean():+.3f}"
      f"   cold books {scores[cold_cols].mean():+.3f}")
print("cold books are pushed low by negative gradient alone -- the real fix is Unit 9's item features.")

## Where this goes

- **Matrix factorization** models the reader x book score table as a product $\mathbf{P}\,\mathbf{Q}^{\top}$ of low-dimensional **latent factors** -- a handful of hidden taste dials per reader and per book, scored by a dot product.
- We **learn** the factors by full-batch, per-entity-averaged **gradient descent** on a **logistic implicit-feedback loss** with L2 regularization -- *trained*, not decomposed, because plain SVD would wrongly force every unobserved pair to zero.
- The negatives decide everything: sample them from each reader's **unobserved complement**, not the log's `label == 0` **exposure** negatives, which carry exposure bias and collapse MF to the random floor (your first brush with exposure bias -> Unit 13).
- On this data MF is **on par with item-item CF** (about 0.25-0.28 hit@10, within noise of CF's ~0.25; here 0.276), far above lexical and popularity -- the latent generalization of co-occurrence, compressed into 32 numbers per entity.

### The bridge to Part 2

The decisive takeaway is structural: `MatrixFactorizationPath.retrieve` is a **dot-product retriever over learned embeddings**. A reader vector, a book vector, a dot product, nearest-neighbour search -- that is the skeleton of every modern learned recommender. **Unit 8** keeps this skeleton exactly and replaces the hand-written training loop with a neural **two-tower** model that learns richer embeddings; **Unit 9** gives cold items feature-based vectors so they are no longer shaped by negative gradient alone; and **Unit 13** takes up the exposure bias we met in the negatives beat. Part 1 built retrieval paths; Part 2 learns them. Matrix factorization is the hinge between the two.